In [ ]:
import re
import os
import json
import cortex
import pickle
import numpy as np
import pandas as pd
import nibabel as nib
from collections import defaultdict
import matplotlib.pyplot as plt


In [ ]:
# TODO - might use the brain plots to show where nuisance regressor has the biggest influence

In [ ]:
# load atlases
# read in atlases
# atlas_base_path = "/home/zachkaras/fmri/fmri_model/analysis/pipeline/atlases"
atlas_base_path = "/home/zachkaras/fmri_model/analysis/pipeline/atlases"

# read in 2d mni mask
mask = nib.load(f"{atlas_base_path}/MNI152_T1_2mm_brain_mask.nii.gz")
og_shape = mask.shape
mask = mask.get_fdata().flatten()
brain_idx = np.where(mask>0)[0]

atlas = nib.load(f"{atlas_base_path}/Schaefer2018_400Parcels_7Networks_order_FSLMNI152_2mm.nii.gz")
atlas_vec = atlas.get_fdata().flatten()
atlas_only_brain = atlas_vec[brain_idx] # contains the schaefer parcel numbers
cortex_vx = np.where(atlas_only_brain != 0)[0]
parcel_nums = atlas_only_brain[cortex_vx]

# Making empty templates to save output
empty_schaefer = np.zeros(atlas_only_brain.shape)
empty_mni = np.zeros(atlas_vec.shape)

# demo_data = pd.read_csv("/home/zachkaras/fmri/fmri_model/master-survey-data.csv")
demo_data = pd.read_csv("/home/zachkaras/fmri_model/master-survey-data.csv")


def convert_to_nifti(values):
    # working backwards to save correlation values as voxels in MNI space
    empty_schaefer[cortex_vx] = values
    empty_mni[brain_idx] = empty_schaefer
    result_brain = np.reshape(empty_mni, og_shape)

    # Saving results
    nifti_result = nib.Nifti1Image(result_brain, affine=atlas.affine, header=atlas.header)
    nib.save(nifti_result, "test_plotting.nii.gz")
    return result_brain, nifti_result

In [ ]:
# brain plots
# participants 117 and 204
# maybe codegemma 7b?
base = "/data/zachkaras/fmri_model_data/ridge_regression_pca_params"
participant = '117'
# participant = '121'
# participant = '204'
filepath = f"{base}/{participant}"
# pattern = r'(?=.*codegemma_7b)(?=.*ndelays_4)(?=.*look_ahead_by_10)'
pattern = r'(?=.*codegemma_7b)(?=.*ndelays_16)(?=.*look_ahead_by_0)'
# param1 = r'(?=.*ndelays_4)(?=.*look_ahead_by_10)'
# param2 = r'(?=.*ndelays_16)(?=.*look_ahead_by_0)'
brain_files = [f for f in os.listdir(filepath) if re.search(pattern, f) and re.search("correlations", f) and re.search("-code-", f)]

In [ ]:
for brain in brain_files:
    print(brain)
    brainpath = f"{filepath}/{brain}"
    with open(brainpath, 'rb') as f:
        data = pickle.load(f)

    npy_brain, nifti_brain = convert_to_nifti(data)
    npy_brain = npy_brain.transpose(2,1,0)
    
    vol = cortex.Volume(
    npy_brain,
    subject='fsaverage',
    xfmname='mni2py2',
    )
    cortex.webshow(vol)
    # break

In [ ]:
vol = cortex.Volume(
    npy_brain,
    subject='fsaverage',
    xfmname='mni2py2',
)
cortex.webshow(vol)